# Advanced challenge: quantum-selected configuration interaction

A fixed variational ansatz can be hard to optimize, especially for strongly correlated geometries. **Quantum-selected configuration interaction (QSCI)** uses a quantum circuit to sample electronic configurations, then asks a classical eigensolver to diagonalize the molecular Hamiltonian in the span of the sampled configurations.

This notebook demonstrates that workflow on square H₄ with a stretched 2.0 Å side length in STO-3G. The UCCSD circuit is deliberately left at a small, unoptimized starting point so we can isolate the post-processing step. The example uses an exact statevector and a dense matrix on a tiny system; it is a teaching demonstration, not a claim of quantum advantage or large-molecule scalability.

**Questions:** How does the selected-subspace energy improve as more configurations are added? How many sampled configurations are needed to reproduce the exact answer for this model?

## The hybrid workflow

1. Prepare a shallow Hartree–Fock + UCCSD trial state.
2. Measure it in the computational basis and retain determinants with the target electron and spin counts.
3. Build the Hamiltonian matrix in the span of those determinants and diagonalize it classically.
4. Compare with Hartree–Fock, the trial-state expectation value, and a particle-sector-filtered exact diagonalization.

The selected-subspace energy is variational: adding configurations can lower it or leave it unchanged, but it cannot go below the exact ground-state energy in the same sector. Sampling does not estimate Hamiltonian matrix elements here; the molecular integrals are used classically to build them.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qiskit.quantum_info import Statevector
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.units import DistanceUnit
from qiskit_algorithms import NumPyMinimumEigensolver

In [ ]:
SIDE_ANGSTROM = 2.0
BASIS = "sto3g"

atom = (
    f"H 0 0 0; H 0 0 {SIDE_ANGSTROM}; "
    f"H 0 {SIDE_ANGSTROM} 0; H 0 {SIDE_ANGSTROM} {SIDE_ANGSTROM}"
)
problem = PySCFDriver(
    atom=atom,
    basis=BASIS,
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
).run()

mapper = JordanWignerMapper()
qubit_hamiltonian = mapper.map(problem.hamiltonian.second_q_op())
n_spatial_orbitals = problem.num_spatial_orbitals
n_alpha, n_beta = problem.num_particles
full_sector_dimension = math.comb(n_spatial_orbitals, n_alpha) * math.comb(
    n_spatial_orbitals, n_beta
)

print(f"Geometry: square H4, side = {SIDE_ANGSTROM:.1f} Å")
print(f"Basis: {BASIS}; electrons (alpha, beta): {problem.num_particles}")
print(f"Qubits: {qubit_hamiltonian.num_qubits}")
print(f"Determinants in the target spin sector: {full_sector_dimension}")

## 1. Establish trustworthy references

The exact solver is filtered to the electron and spin sector requested by the molecular problem. This is important: an unconstrained qubit Hamiltonian can have a lower eigenvalue in an unphysical particle-number sector.

In [ ]:
exact_solver = NumPyMinimumEigensolver(
    filter_criterion=problem.get_default_filter_criterion()
)
exact_result = GroundStateEigensolver(mapper, exact_solver).solve(problem)
exact_energy = float(exact_result.total_energies[0])

hf_state = HartreeFock(n_spatial_orbitals, problem.num_particles, mapper)
hf_statevector = Statevector(hf_state)
nuclear_energy = float(problem.nuclear_repulsion_energy)
hamiltonian_matrix = qubit_hamiltonian.to_matrix()
hf_energy = float(np.real(hf_statevector.expectation_value(qubit_hamiltonian)) + nuclear_energy)

print(f"Hartree-Fock energy: {hf_energy:.10f} Ha")
print(f"Exact target-sector energy: {exact_energy:.10f} Ha")
print(f"Correlation energy beyond Hartree-Fock: {hf_energy - exact_energy:.6f} Ha")

## 2. Prepare and sample a shallow trial circuit

We give UCCSD reproducible, small starting amplitudes instead of running a lengthy optimizer. Its energy is therefore an **unoptimized trial energy**, not a converged VQE result. The purpose is to show how classical subspace diagonalization can improve on the energy of the sampled state.

In [ ]:
ansatz = UCCSD(
    n_spatial_orbitals,
    problem.num_particles,
    mapper,
    initial_state=hf_state,
)
rng = np.random.default_rng(42)
initial_parameters = rng.uniform(-0.1, 0.1, ansatz.num_parameters)
trial_circuit = ansatz.assign_parameters(initial_parameters)
trial_state = Statevector(trial_circuit)
trial_energy = float(
    np.real(trial_state.expectation_value(qubit_hamiltonian)) + nuclear_energy
)

shots = 10_000
trial_state.seed(5)
counts = trial_state.sample_counts(shots=shots)

def in_target_spin_sector(bitstring):
    """Check alpha and beta occupations for Qiskit Nature's JW mode ordering."""
    basis_index = int(bitstring, 2)
    alpha_count = sum((basis_index >> qubit) & 1 for qubit in range(n_spatial_orbitals))
    beta_count = sum(
        (basis_index >> qubit) & 1
        for qubit in range(n_spatial_orbitals, 2 * n_spatial_orbitals)
    )
    return alpha_count == n_alpha and beta_count == n_beta

valid_counts = {
    int(bitstring, 2): int(count)
    for bitstring, count in counts.items()
    if in_target_spin_sector(bitstring)
}
ranked_configurations = sorted(
    valid_counts.items(), key=lambda item: (-item[1], item[0])
)

print(f"Unoptimized UCCSD trial energy: {trial_energy:.10f} Ha")
print(f"Measurement shots: {shots:,}")
print(f"Distinct valid configurations sampled: {len(ranked_configurations)}")
print(f"Full target-sector dimension: {full_sector_dimension}")

## 3. Diagonalize in the sampled subspace

Each selected computational-basis state is one electronic determinant. We take the Hamiltonian submatrix for the most frequently sampled determinants and solve its smallest eigenvalue. The nuclear repulsion is added back to report total molecular energies.

In [ ]:
def qsci_energy(number_of_configurations):
    selected = ranked_configurations[:number_of_configurations]
    basis_indices = [index for index, _ in selected]
    subspace_hamiltonian = hamiltonian_matrix[np.ix_(basis_indices, basis_indices)]
    lowest_electronic_energy = np.linalg.eigvalsh(subspace_hamiltonian)[0]
    return float(lowest_electronic_energy + nuclear_energy)

candidate_sizes = sorted(
    set([1, 4, 8, 12, 16, 20, 24, 28, len(ranked_configurations)])
)
results = pd.DataFrame(
    {
        "sampled_configurations": candidate_sizes,
        "qsci_energy_ha": [qsci_energy(size) for size in candidate_sizes],
    }
)
results["error_to_exact_mha"] = 1000 * (results["qsci_energy_ha"] - exact_energy)
results

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    results["sampled_configurations"],
    results["qsci_energy_ha"],
    "o-",
    label="QSCI selected-subspace energy",
)
plt.axhline(exact_energy, color="black", ls="--", label="Exact target-sector energy")
plt.axhline(hf_energy, color="gray", ls=":", label="Hartree–Fock energy")
plt.axhline(trial_energy, color="tab:orange", ls="-.", label="Unoptimized UCCSD trial")
plt.xlabel("Number of sampled determinants")
plt.ylabel("Total energy (Hartree)")
plt.title("QSCI convergence for stretched square H₄")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

final_qsci_energy = float(results.iloc[-1]["qsci_energy_ha"])
print(f"Selected configurations: {len(ranked_configurations)} / {full_sector_dimension}")
print(f"QSCI energy: {final_qsci_energy:.10f} Ha")
print(f"Absolute error to exact: {abs(final_qsci_energy - exact_energy):.3e} Ha")

## What the result means—and what it does not

For this seeded run, QSCI can recover the exact target-sector energy from a subset of the 36 possible spin-balanced determinants, even though the unoptimized UCCSD trial state has a much higher expectation energy. The selected-space energy improves as configurations are added because each larger space contains the previous one.

This is not evidence that QSCI is cheaper than exact classical chemistry for general systems. Here, we deliberately form a dense 256 × 256 qubit Hamiltonian matrix and the complete 36-determinant sector is already small. For a larger molecule, one would avoid dense matrices, compute selected Hamiltonian elements from integrals, tune shot and determinant-selection budgets, and compare against a well-optimized VQE baseline. On hardware, sampling noise, circuit errors, and missed important determinants can all raise the result.

### Further experiments
- Increase or decrease `shots` and compare the selected configurations and final error.
- Optimize the UCCSD parameters before sampling, then compare QSCI with that converged VQE result.
- Change the square side length and track how strong correlation affects the selected-space size.
- Set a fixed determinant budget and compare frequency ranking with other selection rules.